In [24]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime


In [25]:
# For test purposes

csvFile = ["MSFT", "JNJ", "NVDA","GOOG"]

# A function that grabs the sectors of all the stocks
# and displays them into a data frame
def pull_stock_sectors (tickers):
    sectors = []

    for ticker in tickers:
        stock = yf.Ticker(ticker)
        sector = stock.info.get("sector")

        sectors.append({
            "Ticker": ticker,
            "Sector" :sector
        })
    return pd.DataFrame(sectors)


sector_data = pull_stock_sectors(csvFile)
sector_data

,Ticker,Sector
0,MSFT,Technology
1,JNJ,Healthcare
2,NVDA,Technology
3,GOOG,Communication Services


For this next part, the way I want to choose which sectors are best for each situation is through a scoring system. Where we rank each sector 1 to 5, with 1 being safe and 5 being risky. This will make it a lot easier later on to know which sectors we want to invest in for our portfolio.

In [26]:
# Assigned scores for each sector
sector_scores = {
    "Utilities": 1,
    "Consumer Defensive": 1,
    "Healthcare": 2,
    "Real Estate": 2,
    "Financial Services": 3,
    "Industrials": 3,
    "Energy": 4,
    "Basic Materials": 4,
    "Communication Services": 4,
    "Consumer Cyclical": 4,
    "Technology": 5
}

# Calculates the historical risk of each sector
def calculate_sector_risk(sector_data, prices):
    results = []

    for sector in sector_data["Sector"].dropna().unique():
        tickers = sector_data[
            sector_data["Sector"] == sector
        ]["Ticker"].tolist()

        sector_prices = prices[tickers].dropna()

        # Need at least 30 trading days of data
        if len(sector_prices) < 30:
            continue

        # Equal initial investment in each stock
        stock_values = sector_prices / sector_prices.iloc[0]
        portfolio_value = stock_values.sum(axis=1)

        # Calculate daily returns and risk
        returns = portfolio_value.pct_change().dropna()
        risk = returns.std()

        results.append({
            "Sector": sector,
            "Sector_Risk": risk,
            "Stock_Count": len(tickers)
        })

    return pd.DataFrame(
        results, columns=["Sector", "Sector_Risk", "Stock_Count"]
    )


# Calculates the total portfolio weight in each sector
def calculate_sector_weights(portfolio):
    sector_weights = portfolio.groupby("Sector")["Weight"].sum()
    return sector_weights


# Ranks sectors from lowest to highest historical risk
# This is used with the new updates scoring system using the standard deviation portfolio strategy learned in class
def score_stock_sectors(sector_data, sector_risk):
    sector_risk = sector_risk.copy()

    sector_risk["Risk_Rank"] = sector_risk["Sector_Risk"].rank(
        method="min"
    )

    return sector_data.merge(sector_risk, on="Sector", how="left")

# Checks if any sector exceeds the 40% limit we have
def check_sector_limit(portfolio):
    sector_weights = calculate_sector_weights(portfolio)
    return (sector_weights <= 0.40 + 0.0000000001).all()


Under this is just me testing with a random input to see if the code is doing what its supposed to be doing

In [27]:
# for testing purposes
test_tickers = ["MSFT", "JNJ", "PG", "NVDA", "JPM", "NEE"]

# Get the sectors
sector_data = pull_stock_sectors(test_tickers)

# Download historical prices
prices = yf.download(
    test_tickers,
    start="2026-05-01",
    end="2026-11-18",
    auto_adjust=False
)["Close"]

# Convert USD prices to CAD
exchange_rate = yf.download(
    "CAD=X",
    start="2026-05-01",
    end="2026-11-18"
)["Close"].squeeze()

prices = prices.mul(exchange_rate.reindex(prices.index).ffill(), axis=0)

# Calculate and rank sector risk
sector_risk = calculate_sector_risk(sector_data, prices)
sector_data = score_stock_sectors(sector_data, sector_risk)

display(sector_data)


# Test 1: Technology exceeds 40%
sector_data["Weight"] = [0.25, 0.10, 0.10, 0.25, 0.15, 0.15]

display(calculate_sector_weights(sector_data))
print("Test 1 (should be False):", check_sector_limit(sector_data))


# Test 2: Technology is exactly 40%
sector_data["Weight"] = [0.20, 0.15, 0.15, 0.20, 0.15, 0.15]

print("Test 2 (should be True):", check_sector_limit(sector_data))


# Test 3: All sectors are below 40%
sector_data["Weight"] = [0.15, 0.15, 0.15, 0.15, 0.20, 0.20]

print("Test 3 (should be True):", check_sector_limit(sector_data))


# Check that every stock received a risk ranking
print("Missing risk scores:", sector_data["Risk_Rank"].isna().sum())


[*********************100%***********************]  6 of 6 completed
[*********************100%***********************]  1 of 1 completed


,Ticker,Sector,Sector_Risk,Stock_Count,Risk_Rank
0,MSFT,Technology,0.018690,2,5.0
1,JNJ,Healthcare,0.015688,1,4.0
2,PG,Consumer Defensive,0.014033,1,3.0
3,NVDA,Technology,0.018690,2,5.0
4,JPM,Financial Services,0.013977,1,2.0
5,NEE,Utilities,0.013207,1,1.0


Sector
Consumer Defensive    0.10
Financial Services    0.15
Healthcare            0.10
Technology            0.50
Utilities             0.15
Name: Weight, dtype: float64

Test 1 (should be False): False
Test 2 (should be True): True
Test 3 (should be True): True
Missing risk scores: 0
